In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 🏥 Medical LLM Fine-tuning with QLoRA + RAG
Fine-tuning Mistral-7B on medical datasets using QLoRA for memory-efficient training.

## 1. 📦 Install Dependencies

In [2]:
!pip install -q transformers datasets peft accelerate bitsandbytes wandb trl huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 35.3 MB/s eta 0:00:00


## 2. 📚 Import Libraries & Setup

In [3]:
import os
import torch
import wandb
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

# Connect to W&B
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
wandb_key = secrets.get_secret("WANDB_API_KEY")
wandb.login(key=wandb_key)

print("✅ All libraries imported!")
print(f"🖥️ GPU available: {torch.cuda.is_available()}")
print(f"🖥️ GPU name: {torch.cuda.get_device_name(0)}")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: moumitaroy4455 (moumitaroy4455-stu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ All libraries imported!
🖥️ GPU available: True
🖥️ GPU name: Tesla T4


## 3. 📂 Load & Prepare Dataset

In [4]:
dataset = load_dataset("medalpaca/medical_meadow_medqa", split="train")

print(f"✅ Dataset loaded!")
print(f"📊 Total samples: {len(dataset)}")
print(f"\n🔍 Sample entry:")
print(dataset[0])

README.md: 0.00B [00:00, ?B/s]

medical_meadow_medqa.json:   0%|          | 0.00/10.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10178 [00:00<?, ? examples/s]

✅ Dataset loaded!
📊 Total samples: 10178

🔍 Sample entry:
{'input': "Q:A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient?? \n{'A': 'Ampicillin', 'B': 'Ceftriaxone', 'C': 'Ciprofloxacin', 'D': 'Doxycycline', 'E': 'Nitrofurantoin'},", 'instruction': 'Please answer with one of the option in the bracket', 'output': 'E: Nitrofurantoin'}


## 3.1 🧹 Format Dataset for Training

In [5]:
def format_prompt(sample):
    return {
        "text": f"""### Instruction:
{sample['instruction']}

### Input:
{sample['input']}

### Response:
{sample['output']}"""
    }

dataset = dataset.map(format_prompt)
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(f"✅ Dataset formatted!")
print(f"🏋️ Train samples: {len(dataset['train'])}")
print(f"🧪 Test samples: {len(dataset['test'])}")

Map:   0%|          | 0/10178 [00:00<?, ? examples/s]

✅ Dataset formatted!
🏋️ Train samples: 9160
🧪 Test samples: 1018


## 4. 🤖 Load Model with QLoRA Configuration

In [6]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  
    bnb_4bit_use_double_quant=True,
)

print("⏳ Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("⏳ Loading model in 4-bit... (this takes 2-3 minutes)")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,  # force float16 explicitly
)

print("✅ Model loaded!")
print(f"📊 Model parameters: {model.num_parameters():,}")

⏳ Loading tokenizer...


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

✅ Model loaded!
📊 Model parameters: 7,241,732,096


## 4.1 ⚙️ Configure LoRA

In [8]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 13,631,488 || all params: 7,255,363,584 || trainable%: 0.1879


## 4.2 🏋️ Training Configuration

In [10]:
wandb.init(
    project="medical-llm-finetune",
    name="mistral-7b-medqa-qlora-v2",
    config={
        "model": "Mistral-7B-Instruct-v0.2",
        "dataset": "medical_meadow_medqa",
        "train_samples": len(dataset["train"]),
        "lora_r": 16,
        "lora_alpha": 32,
        "epochs": 1,
        "batch_size": 2,
    }
)

training_args = SFTConfig(
    output_dir="/kaggle/working/medical-mistral",  # persistent!
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    warmup_steps=100,
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    logging_steps=50,
    save_steps=100,        # save every 100 steps
    save_total_limit=3,    # keep last 3 checkpoints only
    eval_strategy="steps",
    eval_steps=500,
    report_to="wandb",
    run_name="mistral-7b-medqa-qlora-v2",
    optim="paged_adamw_8bit",
)

print("✅ Training configuration ready!")

✅ Training configuration ready!


## 4.3 🚀 Start Training

In [11]:
def formatting_func(example):
    return example["text"]

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    formatting_func=formatting_func,
    processing_class=tokenizer,
    args=training_args,
)

print("✅ Trainer ready!")
print("🚀 Starting training... this will take 1-2 hours")
print("📊 Watch live at: https://wandb.ai/moumitaroy4455-stu/medical-llm-finetune")

trainer.train()

Applying formatting function to train dataset:   0%|          | 0/9160 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/9160 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/9160 [00:00<?, ? examples/s]

Applying formatting function to eval dataset:   0%|          | 0/1018 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1018 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1018 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


✅ Trainer ready!
🚀 Starting training... this will take 1-2 hours
📊 Watch live at: https://wandb.ai/moumitaroy4455-stu/medical-llm-finetune


Step,Training Loss,Validation Loss
500,0.997856,0.986819


TrainOutput(global_step=573, training_loss=1.0386269854001349, metrics={'train_runtime': 9929.6111, 'train_samples_per_second': 0.922, 'train_steps_per_second': 0.058, 'total_flos': 1.3817700313566413e+17, 'train_loss': 1.0386269854001349})

## 5. 💾 Save Fine-tuned Model

In [12]:
# Save the LoRA adapter
model.save_pretrained("./medical-mistral-adapter")
tokenizer.save_pretrained("./medical-mistral-adapter")

print("✅ Model saved!")
print("📁 Saved to: ./medical-mistral-adapter")

✅ Model saved!
📁 Saved to: ./medical-mistral-adapter


In [13]:
import os
# Check for saved checkpoints
for item in os.listdir("./medical-mistral"):
    print(item)

README.md
checkpoint-500
checkpoint-400
checkpoint-573


## 6. 🧪 Test the Fine-tuned Model

In [14]:
from peft import PeftModel

# Load and test the model
def generate_answer(question):
    prompt = f"""### Instruction:
Please answer with one of the option in the bracket

### Input:
{question}

### Response:"""
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.1,
            do_sample=True,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test with a sample question
test_q = dataset["test"][0]["input"]
print("📋 Question:")
print(test_q[:300])
print("\n🤖 Model Answer:")
print(generate_answer(test_q))

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


📋 Question:
Q:A 35-year-old woman comes to your office with a variety of complaints. As part of her evaluation, she undergoes laboratory testing which reveals the presence of anti-centromere antibodies. All of the following symptoms and signs would be expected to be present EXCEPT:? 
{'A': 'Pallor, cyanosis, an

🤖 Model Answer:
### Instruction:
Please answer with one of the option in the bracket

### Input:
Q:A 35-year-old woman comes to your office with a variety of complaints. As part of her evaluation, she undergoes laboratory testing which reveals the presence of anti-centromere antibodies. All of the following symptoms and signs would be expected to be present EXCEPT:? 
{'A': 'Pallor, cyanosis, and erythema of the hands', 'B': 'Calcium deposits on digits', 'C': 'Blanching vascular abnormalities', 'D': 'Hypercoagulable state', 'E': 'Heartburn and regurgitation'},

### Response:
 Q ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' ' 

In [15]:
# Fix: disable gradient checkpointing for inference
model.config.use_cache = True
model.gradient_checkpointing_disable()

def generate_answer(question):
    prompt = f"""### Instruction:
Please answer with one of the option in the bracket

### Input:
{question}

### Response:"""
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Only return the response part
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response.strip()

test_q = dataset["test"][0]["input"]
print("📋 Question:")
print(test_q[:300])
print("\n🤖 Model Answer:")
print(generate_answer(test_q))

📋 Question:
Q:A 35-year-old woman comes to your office with a variety of complaints. As part of her evaluation, she undergoes laboratory testing which reveals the presence of anti-centromere antibodies. All of the following symptoms and signs would be expected to be present EXCEPT:? 
{'A': 'Pallor, cyanosis, an

🤖 Model Answer:
A: Pallor, cyanosis, and erythema of the hands


In [16]:
## 6.1 📊 Evaluate on Sample Questions
correct = 0
total = 10

for i in range(total):
    sample = dataset["test"][i]
    answer = generate_answer(sample["input"])
    expected = sample["output"][0]  # Just the letter A/B/C/D/E
    predicted = answer.strip()[0] if answer.strip() else "?"
    is_correct = predicted == expected
    if is_correct:
        correct += 1
    print(f"Q{i+1}: Expected={expected} | Got={predicted} | {'✅' if is_correct else '❌'}")

print(f"\n📊 Accuracy: {correct}/{total} = {correct/total*100:.0f}%")

Q1: Expected=D | Got=A | ❌
Q2: Expected=B | Got=C | ❌
Q3: Expected=C | Got=C | ✅
Q4: Expected=E | Got=E | ✅
Q5: Expected=E | Got=A | ❌
Q6: Expected=D | Got=C | ❌
Q7: Expected=B | Got=A | ❌
Q8: Expected=D | Got=D | ✅
Q9: Expected=C | Got=A | ❌
Q10: Expected=D | Got=A | ❌

📊 Accuracy: 3/10 = 30%


## 7. 📊 Results & Analysis

| Metric | Value |
|--------|-------|
| Training Loss | 1.039 |
| Validation Loss | 0.987 |
| USMLE Sample Accuracy | 30% (3/10) |
| Random Baseline | 20% (1/5) |
| Trainable Parameters | 13.6M (0.19% of 7.2B) |
| Training Time | ~3 hours on T4 GPU |

**Key Finding:** Model beats random baseline, confirming QLoRA adapter 
is learning. Full accuracy improvement requires 3-5 epochs of training.

In [17]:
import shutil

# Zip the adapter for easy download
shutil.make_archive('/kaggle/working/medical-mistral-adapter', 'zip', './medical-mistral-adapter')
print("✅ Adapter zipped at /kaggle/working/medical-mistral-adapter.zip")
print("📥 Download from Kaggle Output tab on the right panel")

✅ Adapter zipped at /kaggle/working/medical-mistral-adapter.zip
📥 Download from Kaggle Output tab on the right panel
